# Step 2: Feature Extraction & Processing
## Paper: Emergent risk asymmetry in high-dimensional multi-agent self-play


---

**Author:** Kenny Ching  
**Affiliation:** School of Business, University of Auckland  
**Date:** February 2026  

---

**Description:**
This notebook parses the raw JSON replay files downloaded in Step 1. It reconstructs the economic timeline of every match to generate the event-level dataset used for regression analysis.

**Key Calculations:**
* **Teamfight Detection:** Identifies combat events using the Source 2 combat log.
* **Strategic Yield ($\Delta G_{180s}$):** Calculates the change in Net Worth Advantage exactly 180 seconds post-fight.
* **Tactical Outcome:** Determines the net unit loss/gain (Kill Delta) for the Radiant team.

**Input:**
* Raw JSON files from `data/raw/`.

**Output:**
* `data/processed/tf_events_final.csv` (The master dataset).

In [ ]:
import os
import glob
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from google.colab import drive
from google.colab import files

# ==========================================
# 1. SETUP
# ==========================================
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

# PATH CONFIGURATION
DATA_DIR = "/content/drive/My Drive/OpenAI_Data/data"
OUTPUT_FILE = "tf_events_final-1.csv"
DRIVE_SAVE_PATH = os.path.join(DATA_DIR, OUTPUT_FILE)

OPENAI_IDS = [4693543125, 4693543123, 4693543117, 4080601137, 4080420268, 4080316101]

# ==========================================
# 2. PARSING LOGIC (FIXED)
# ==========================================
def get_gold_at_time(gold_adv_list, time_sec):
    if not gold_adv_list: return 0
    idx = int(time_sec / 60)
    if idx < len(gold_adv_list):
        return gold_adv_list[idx]
    return gold_adv_list[-1]

def parse_match(file_path):
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
    except Exception:
        return []

    match_id = data.get('match_id')
    if not match_id: return []

    is_openai = match_id in OPENAI_IDS
    gold_adv = data.get('radiant_gold_adv', [])
    if not gold_adv: return []

    events = []

    for tf in data.get('teamfights', []):
        start = tf['start']

        # Calculate Kill Delta
        radiant_kills = 0
        dire_kills = 0
        for i, p in enumerate(tf['players']):
            deaths = p['deaths']
            if i < 5:
                dire_kills += deaths
            else:
                radiant_kills += deaths
        kill_delta = radiant_kills - dire_kills

        # Calculate Strategic Yields (FIXED: Added g_60)
        base_g = get_gold_at_time(gold_adv, start)
        g_30 = get_gold_at_time(gold_adv, start + 30) - base_g
        g_60 = get_gold_at_time(gold_adv, start + 60) - base_g  # <-- Added this
        g_120 = get_gold_at_time(gold_adv, start + 120) - base_g
        g_180 = get_gold_at_time(gold_adv, start + 180) - base_g

        events.append({
            'match_id': match_id,
            'is_openai5': is_openai,
            'fight_start': start,
            'kill_delta_radiant': kill_delta,
            'gadv_d30': g_30,
            'gadv_d60': g_60,   # <-- Included in output
            'gadv_d120': g_120,
            'gadv_d180': g_180
        })
    return events

# ==========================================
# 3. EXECUTION
# ==========================================
if __name__ == "__main__":
    print(f"Scanning for JSON files in: {DATA_DIR}")
    files_list = glob.glob(os.path.join(DATA_DIR, "**", "*.json"), recursive=True)
    print(f"Found {len(files_list)} JSON files.")

    if len(files_list) > 0:
        all_events = []
        print("Processing matches (Extraction phase)...")
        for f in tqdm(files_list):
            all_events.extend(parse_match(f))

        if all_events:
            df = pd.DataFrame(all_events)

            # Save to Drive (Overwriting the old incomplete file)
            df.to_csv(DRIVE_SAVE_PATH, index=False)
            print(f"\n✅ SUCCESS: Regenerated {DRIVE_SAVE_PATH}")
            print(f"Total Events: {len(df)}")
            print("Columns:", df.columns.tolist())

            # Helper download
            files.download(DRIVE_SAVE_PATH)
        else:
            print("No teamfights extracted.")
    else:
        print("No files found. Check path.")

Scanning for JSON files in: /content/drive/My Drive/OpenAI_Data/data
Found 2287 JSON files.
Processing matches (Extraction phase)...


100%|██████████| 2287/2287 [00:32<00:00, 71.17it/s]



✅ SUCCESS: Regenerated /content/drive/My Drive/OpenAI_Data/data/tf_events_final-1.csv
Total Events: 12422
Columns: ['match_id', 'is_openai5', 'fight_start', 'kill_delta_radiant', 'gadv_d30', 'gadv_d60', 'gadv_d120', 'gadv_d180']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ... (after df = pd.DataFrame(all_events)) ...

# Define the Drive path
DRIVE_PATH = os.path.join(DATA_DIR, "tf_events_final-1.csv")

# Save directly to Google Drive
df.to_csv(DRIVE_PATH, index=False)
print(f"✅ Saved permanently to Drive: {DRIVE_PATH}")



✅ Saved permanently to Drive: /content/drive/My Drive/OpenAI_Data/data/tf_events_final-1.csv
